# Residual extremes

What happens after a large residual: how long until it crosses zero, how much worse it gets first, and how often it never resolves at all.

**This notebook contains no calculations.** Everything is computed by
`xauusd_quant.features` and `xauusd_quant.research` and read back from
`results/regression_research/<timeframe>/window_<N>/`. Generate those first:

```
xq regression-research --timeframe 1h --window 128
xq regression-research --timeframe 5m --all-windows
xq regression-research --all
```

No trading rule is defined or evaluated anywhere in this notebook. No
transaction costs are applied. Nothing here is a signal.

In [ ]:
import sys
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl

from xauusd_quant import load_config
from xauusd_quant.features import load_regression_config

config = load_config(ROOT / "config" / "data.yaml")
regression = load_regression_config(ROOT / "config" / "regression.yaml")

TIMEFRAME = "1h"          # change and re-run
WINDOW = 128              # change and re-run

print(f"instrument : {config.instrument}")
print(f"timezone   : {config.timezone.describe()}")
print(f"transform  : {regression.regression.price_transform} of "
      f"{regression.regression.price_source} {regression.regression.price_column}")
print(f"windows    : {list(regression.regression.windows)}")
print(f"fingerprint: {regression.fingerprint()}")

In [ ]:
import json

results = regression.results_dir(TIMEFRAME, WINDOW)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME} window={WINDOW}. Run:  "
        f"xq regression-research --timeframe {TIMEFRAME} --window {WINDOW}"
    )

def table(name):
    """Load a previously-generated regression table."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
provenance = summary["provenance"]
model = summary.get("model_summary", {})

print(f"generated  : {provenance['generated_utc']}")
print(f"window     : {provenance['regression_window']} bars on {provenance['timeframe']}")
print(f"valid fits : {provenance['observations']:,}")
print(f"span       : {provenance['requested_start']} -> {provenance['requested_end']}")
print(f"git        : {provenance['git']}")
for warning in summary.get("warnings", []):
    print(f"WARNING: {warning}")

## Time to zero crossing, with censoring made explicit

Starts that never cross inside the search horizon are **not dropped**. They are
counted in `censored_count` and `censored_fraction`, and the median is reported
as null when more than half the starts are censored, because in that case the
median is genuinely unknowable rather than large.

Silently discarding censored starts is the standard way to manufacture a short,
reassuring "time to reversion".

Starts too close to the end of the sample to have a full horizon are excluded
altogether, so that position alone never counts as censoring.

In [ ]:
table('zero_crossing')

## Does it get worse before it gets better?

From a start beyond the threshold, this is the probability that `|Z|` reaches
each higher level **before** falling back. High values mean the residual
routinely overshoots on its way to reverting - which is exactly what an
"it mean-reverts" summary hides.

In [ ]:
table('extreme_persistence')

## Maximum adverse residual excursion

MARE is how much further from zero the residual travelled within the horizon,
in residual units and in Z units.

It is **not** trading MAE: there is no position, no entry price, no currency
and no cost. It measures the series, not a trade.

In [ ]:
table('adverse_excursion')

## Intraday

Session and hour-of-day behaviour of the residual, on the configured time
basis. The timezone shown in the setup cell is the one these hours refer to.

In [ ]:
table('intraday')

## Closing note

Everything above describes the *distribution* of a detrended price series. To
become a trading claim it would need, at minimum: a rule, an execution
assumption that fills at the ask and exits at the bid, the spread subtracted
from every round trip, and an out-of-sample test. None of those exist yet, and
none are in scope for Prompt #3.

In [ ]:
print(model.get("caveat", ""))